In [1]:
import torch
from torch import nn
import pandas as pd
import random
import optuna
from torch.optim.lr_scheduler import OneCycleLR
import re
from nltk.tokenize import word_tokenize
from collections import Counter
from torch import tensor

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [112]:
def format_vectors(vector):
    clean = re.sub(r",?\s*grad_fn=<[^>]+>", "", vector)
    return (eval(clean), {"tensor": torch.tensor})[0]

def clean_sentence(s):
    s = s.strip()

    # Remove a trailing "active" along with the period right before it
    s = re.sub(r'\.\s*active\s*$', '', s)

    # # # Remove a trailing "passive" along with the period right before it
    # s = re.sub(r'\.\s*passive\s*$', '', s)

    # Underscores -> spaces
    s = s.replace('_', ' ')

    # Collapse any double spaces left behind and trim
    s = re.sub(r'\s+', ' ', s).strip()

    return s

In [113]:
df = pd.read_csv("pass_quest_H.csv")
df['context'] = df['context'].apply(format_vectors)
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device = torch.device("mps")

print(device)
print(len(df))

mps
9960


In [114]:
# 1. Tokenize each entry in the column
df['input'] = df['input'].apply(clean_sentence)
df['tokens'] = df['input'].apply(word_tokenize)

# 2. Build a vocab dict (token -> integer id)
all_tokens = [tok for tokens in df['tokens'] for tok in tokens]
counts = Counter(all_tokens)

# sort by frequency (most common first) -- optional but common practice
vocab = {tok: i for i, (tok, _) in enumerate(counts.most_common())}

In [125]:
for i in df["input"]:
    print(i)

the grotesque salamander who our exuberant exuberant handsome vulture does high five does high five the walrus who doesn't applaud some newts
the fantastic orangutans who do entertain your determined newt do admire my agreeable vultures who do admire her determined salamander
my courageous newts that our grotesque determined courageous ravens do entertain do accept some xylophone who doesn't remember her salamander . passive
her handsome handsome determined orangutan that our zebras don't confuse doesn't entertain your courageous orangutans that the quails don't remember
your exuberant salamander that your yaks don't remember does entertain some exuberant handsome unicorn who our handsome yak does entertain . passive
her xylophone who her handsome vultures don't accept does high five her quail alongside your courageous zebra
some agreeable zebras above our agreeable salamanders do admire our walruses that don't confuse your ravens . passive
some agreeable handsome tyrannosauruses do ap

In [115]:
df_sorted = df.sort_values(by='tokens', key=lambda col: col.map(len))

model_size = df['context'].iloc[0].shape[1]
print(model_size)

vocab_size = len(vocab)
print(vocab_size)
batch_size = 128

256
65


In [116]:
left_role = 0
right_role = 1
def right_recurse(lst):
    if not lst:
        return []
    
    batch_size = lst[0].shape[0]
    tensors = [[x, torch.full((batch_size,), left_role, device = device)] for x in lst[:-1]]
    
    acc = lst[-1]
    for t in reversed(tensors):
        acc = [t, [acc, torch.full((batch_size,), right_role, device = device)]]
    return acc

def left_recurse(lst):
    if not lst:
        return []
    
    batch_size = lst[0].shape[0]
    tensors = [[x, torch.full((batch_size,), right_role, device = device)] for x in lst[1:]]
    
    acc = lst[0]
    for t in tensors:
        acc = [[acc, torch.full((batch_size,), left_role, device = device)], t]
    return acc


In [117]:
def collate_lists(*lists):
    return [torch.tensor([vocab[lst[i]] for lst in lists]) for i in range(len(lists[0]))]

In [118]:
sources = df_sorted["tokens"].tolist()
targets = df_sorted["context"].tolist()

from collections import defaultdict

buckets = defaultdict(list) 

for src, trg in zip(sources, targets):
    buckets[len(src)].append((src, trg))

In [119]:
src_batches = []
trg_batches = []

for seq_len in sorted(buckets.keys()):
    pairs = buckets[seq_len]
    for i in range(0, len(pairs), batch_size):
        # If i + batch_size is out of bounds, it just goes to the end of the list
        chunk = pairs[i:i + batch_size]
        src_batch = [p[0] for p in chunk]
        trg_batch = [p[1] for p in chunk]

        collated = collate_lists(*src_batch) 
        nested   = right_recurse(collated)
        # nested = left_recurse(collated)

        src_batches.append(nested)
        trg_batches.append(torch.stack([torch.tensor(v, dtype=torch.float32) for v in trg_batch]))

n = len(src_batches)
n_train = int(0.8 * n)
n_val = int(0.1 * n)

combined = list(zip(src_batches, trg_batches))

random.seed(1)
random.shuffle(combined)
src_batches, trg_batches = zip(*combined)

train_src, train_trg = src_batches[:n_train], trg_batches[:n_train]
val_src, val_trg = src_batches[n_train:n_train+n_val], trg_batches[n_train:n_train+n_val]
test_src, test_trg = src_batches[n_train+n_val:], trg_batches[n_train+n_val:]

/var/folders/fm/0srnvghj4vb0yx0pw0j892f80000gn/T/ipykernel_8502/371104976.py:17: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  trg_batches.append(torch.stack([torch.tensor(v, dtype=torch.float32) for v in trg_batch]))


In [120]:
def get_recursive_batch_size(x):
    if isinstance(x[0], list):
        return get_recursive_batch_size(x[0])
    return x[0].shape[0]
    
class RTPREncode(nn.Module):
    def __init__(self, nfillers, nroles, filler_size, role_size, model_size):
        super(RTPREncode, self).__init__()
        self.fillers = nn.Embedding(nfillers, filler_size)
        self.roles = nn.Embedding(nroles, role_size)
        self.filler_size = filler_size
        self.role_size = role_size
        self.model_size = model_size
        # self.dropout = nn.Dropout(0.4)
        self.w = torch.nn.Parameter(torch.randn(filler_size, filler_size*role_size)*0.01)
        self.wo = torch.nn.Parameter(torch.randn(model_size, filler_size)*0.01)

    def forward(self, x):
        cur_batch_size = get_recursive_batch_size(x)
        
        result = torch.zeros(cur_batch_size, self.filler_size, self.role_size, device=self.w.device)
    
        for pair in x:
            filler = pair[0]
            role = pair[1].to(device)
            
            if isinstance(filler, list):
                filler = self(filler)
            else:
                assert filler.to(device).max().item() < self.fillers.num_embeddings, \
                f"max index {filler.to(device).max().item()} >= num_embeddings {self.fillers.num_embeddings}"
                filler = self.fillers(filler.to(device))
            
            role = self.roles(role)
            result += torch.einsum("bf,br->bfr", filler, role)
        
        result = result.view(cur_batch_size, -1)
        result = torch.einsum("ij,bj->bi", self.w, result) 
        return result

In [121]:
#Evaluate a model (for validation and testing loss)
def evaluate(model, src, trg, criterion, device):
    with torch.no_grad():
        model.eval()
        epoch_loss = 0.0

        for i in range(len(src)):
            src_batch = src[i]
            trg_batch = trg[i].squeeze(1).to(device)
            encoding = model(src_batch)
            output = torch.matmul(encoding, model.wo.T)    # (batch_size, model_size)
            loss = criterion(output, trg_batch)
            epoch_loss += loss.item()

        return epoch_loss / len(src)

In [51]:
def objective(trial):
    num_epochs = 50

    model = RTPREncode(vocab_size, 2, 64, 64, model_size).to(device)
    learning_rate = trial.suggest_float("lr", 1e-5, 1e-3, log = True)
    norm_val = trial.suggest_float("grad_norm", 1, 10, log = True)
    max_rate = trial.suggest_float("scheduler_max", 1e-4, 1e-2, log = True)

    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-4)
    criterion = nn.MSELoss()
    scheduler = OneCycleLR(optimizer, max_lr=max_rate, steps_per_epoch=len(train_src), epochs=num_epochs)

    for epoch in range(num_epochs):
        print(f"Epoch {epoch+1} / {num_epochs}")
        # print(f"Trial {trial.number}")
        model.train()

        epoch_loss = 0.0
        with torch.set_grad_enabled(True):
            for i in range(len(train_src)):
                src_batch = train_src[i]
                trg_batch = train_trg[i].to(device)
                optimizer.zero_grad()

                encoding = model(src_batch)
                output = torch.matmul(encoding, model.wo.T)

                loss = criterion(output, trg_batch)
                loss.backward()

                #Norm=magnitude, clipping gradient to ensure they don't explode
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm = norm_val)
                optimizer.step()
                scheduler.step()
                epoch_loss += loss.item()

        print(f"Train Loss: {epoch_loss/len(train_src)}")
        val_loss = evaluate(model, val_src, val_trg, criterion, device)

        trial.report(val_loss, epoch)
        #prune cuts off trial early
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()
        print(f"Validation Loss: {val_loss}")

    return val_loss
    

In [26]:
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=8)
print("Best Hyperparameters:", study.best_params) 

[I 2026-07-06 14:14:50,594] A new study created in memory with name: no-name-f1f75e6f-f415-4f9b-9520-3bc9c39f390f
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/torch/nn/modules/loss.py:626: UserWarning: Using a target size (torch.Size([8, 1, 256])) that is different to the input size (torch.Size([8, 256])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Epoch 1 / 50


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/torch/nn/modules/loss.py:626: UserWarning: Using a target size (torch.Size([128, 1, 256])) that is different to the input size (torch.Size([128, 256])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/torch/nn/modules/loss.py:626: UserWarning: Using a target size (torch.Size([32, 1, 256])) that is different to the input size (torch.Size([32, 256])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/torch/nn/modules/loss.py:626: UserWarning: Using a target size (torch.Size([83, 1, 256])) that is different to the input size (t

Train Loss: 0.40186459343466496
Validation Loss: 0.18211894896295336
Epoch 2 / 50


[W 2026-07-06 14:14:58,333] Trial 0 failed with parameters: {'lr': 0.0002878971943829445, 'grad_norm': 9.749321733194844, 'scheduler_max': 0.0027039643774293483} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
  File "/var/folders/fm/0srnvghj4vb0yx0pw0j892f80000gn/T/ipykernel_8502/3544351886.py", line 38, in objective
    val_loss = evaluate(model, val_src, val_trg, criterion, device)
  File "/var/folders/fm/0srnvghj4vb0yx0pw0j892f80000gn/T/ipykernel_8502/2870929805.py", line 10, in evaluate
    encoding = model(src_batch)
  File "/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/torch/nn/modules/module.py", line 1778, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^
  File "/Library/Fr

Train Loss: 0.16885018969575563


KeyboardInterrupt: 

In [122]:
# Train with optimized hyperparams
num_epochs = 50

model = RTPREncode(vocab_size, 2, 64, 64, model_size).to(device)
learning_rate = 1e-4
norm_val = 5
max_rate = 1e-4

optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-4)
criterion = nn.MSELoss()
scheduler = OneCycleLR(optimizer, max_lr=max_rate, steps_per_epoch=len(train_src), epochs=num_epochs)

for epoch in range(num_epochs):
    print(f"Epoch {epoch+1} / {num_epochs}")
    # print(f"Trial {trial.number}")
    model.train()

    epoch_loss = 0.0
    with torch.set_grad_enabled(True):
        for i in range(len(train_src)):
            src_batch = train_src[i]
            trg_batch = train_trg[i].squeeze(1).to(device)
            optimizer.zero_grad()

            encoding = model(src_batch)
            output = torch.matmul(encoding, model.wo.T)

            loss = criterion(output, trg_batch)
            loss.backward()

            #Norm=magnitude, clipping gradient to ensure they don't explode
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm = norm_val)
            optimizer.step()
            scheduler.step()
            epoch_loss += loss.item()

    print(f"Train Loss: {epoch_loss/len(train_src)}")
    val_loss = evaluate(model, val_src, val_trg, criterion, device)
    print(f"Validation Loss: {val_loss}")

Epoch 1 / 50
Train Loss: 0.5501968603428096
Validation Loss: 0.5259587003125085
Epoch 2 / 50
Train Loss: 0.4787619758958686
Validation Loss: 0.3997375766436259
Epoch 3 / 50
Train Loss: 0.39994078752112716
Validation Loss: 0.28904368148909676
Epoch 4 / 50
Train Loss: 0.45513723096618913
Validation Loss: 0.2257665991783142
Epoch 5 / 50
Train Loss: 1.2156536038199517
Validation Loss: 0.21049498352739546
Epoch 6 / 50
Train Loss: 0.29766420101466246
Validation Loss: 0.18979744778739083
Epoch 7 / 50
Train Loss: 0.18942272703941554
Validation Loss: 0.1747775442070431
Epoch 8 / 50
Train Loss: 0.23817634010968142
Validation Loss: 0.17356236610147688
Epoch 9 / 50
Train Loss: 0.1783120481118764
Validation Loss: 0.20211584203773075
Epoch 10 / 50
Train Loss: 0.20010788452951875
Validation Loss: 0.16335934731695387
Epoch 11 / 50
Train Loss: 0.17616455216113835
Validation Loss: 0.1640615297688378
Epoch 12 / 50
Train Loss: 0.21866756575564816
Validation Loss: 0.1677363912264506
Epoch 13 / 50
Train Los

In [123]:
print(f"Train Loss: {evaluate(model, train_src, train_trg, criterion, device)}")
print(f"Val Loss: {evaluate(model, val_src, val_trg, criterion, device)}")
print(f"Test Loss: {evaluate(model, test_src, test_trg, criterion, device)}")

Train Loss: 0.026370687257141284
Val Loss: 0.030084647238254547
Test Loss: 0.025202134810388088


In [54]:
torch.save(model.state_dict(), "rbt_pp.pt")